In [1]:
import requests
import pandas as pd
import re
import numpy as np

In [2]:
col_sum_match_dict = {
  "TOTPOP" : ['POPULATION'],
  "GQPOP"  : ['GROUP_QUARTERS_POPULATION'],
  "HHPOP"  : ['HOUSEHOLD_POPULATION'],
  "HH"     : ['HOUSEHOLDS'],
  "POP0017": ['POPULATION_AGES_0_4', 'POPULATION_AGES_5_17'],
  "POP1864": ['POPULATION_AGES_18_24', 'POPULATION_AGES_25_64'],
  "POP65P" : ['POPULATION_AGES_65_84', 'POPULATION_AGES_85+']
}

In [3]:
from _counties import get_counties_df
counties_df = get_counties_df()
counties_df

,CO_FIPS,CO_NAME
0,5,CACHE
1,13,DUCHESNE
2,11,DAVIS
3,27,MILLARD
4,51,WASATCH
5,3,BOX ELDER
6,57,WEBER
7,23,JUAB
8,39,SANPETE
9,53,WASHINGTON


# Get GPI population data

In [4]:
pop_hh_df = pd.read_csv("data/GPI-SE-Data-20251121/gpi-demographic-detail.csv")

pop_hh_df.columns = pop_hh_df.columns.str.upper().str.replace(" ","_").str.replace("-","_")

pop_hh_df['CO_NAME'] = pop_hh_df['GEOGRAPHY'].str.replace(" County","").str.upper()

pop_hh_df = pop_hh_df[pop_hh_df['CO_NAME']!="UTAH STATE"]

pop_hh_df = pop_hh_df.merge(counties_df[['CO_NAME','CO_FIPS']], on='CO_NAME', how='outer')

pop_hh_df

,GEOGRAPHY,YEAR,POPULATION,BIRTHS,DEATHS,NATURAL_CHANGE,NET_MIGRATION,POPULATION_AGES_0_4,POPULATION_AGES_5_17,POPULATION_AGES_18_24,POPULATION_AGES_25_64,POPULATION_AGES_65_84,POPULATION_AGES_85+,HOUSEHOLDS,PERSONS_PER_HOUSEHOLD,HOUSEHOLD_POPULATION,GROUP_QUARTERS_POPULATION,MEDIAN_AGE,CO_NAME,CO_FIPS
0,Beaver County,2025,7533,92,59,33,161,513,1558,855,3455,1075,77,2472,2.90,7163,369,34.65,BEAVER,1
1,Beaver County,2026,7577,93,60,32,12,514,1516,907,3446,1112,82,2516,2.87,7226,351,34.82,BEAVER,1
2,Beaver County,2027,7582,92,61,31,-26,494,1502,870,3493,1136,87,2561,2.85,7287,295,34.92,BEAVER,1
3,Beaver County,2028,7634,92,62,30,22,483,1488,870,3516,1182,95,2600,2.82,7331,303,35.10,BEAVER,1
4,Beaver County,2029,7674,91,62,29,10,474,1472,847,3552,1228,101,2636,2.79,7363,311,35.24,BEAVER,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1184,Weber County,2061,373889,3091,3927,-836,3807,15828,45940,29498,194304,74198,14121,163670,2.25,368298,5591,46.41,WEBER,57
1185,Weber County,2062,376857,3112,3994,-881,3849,15898,45627,29586,195648,75365,14733,165585,2.24,371211,5646,46.69,WEBER,57
1186,Weber County,2063,379822,3136,4061,-925,3890,15988,45339,29619,197111,76383,15382,167492,2.23,374122,5701,46.97,WEBER,57
1187,Weber County,2064,382790,3162,4129,-967,3934,16095,45089,29599,198790,77161,16056,169387,2.23,377033,5757,47.22,WEBER,57


In [5]:
for new_col, source_cols in col_sum_match_dict.items():
    pop_hh_df[new_col] = pop_hh_df[source_cols].sum(axis=1)

pop_hh_df = pop_hh_df[["CO_FIPS","YEAR","CO_NAME"] + list(col_sum_match_dict.keys())]

pop_hh_df

,CO_FIPS,YEAR,CO_NAME,TOTPOP,GQPOP,HHPOP,HH,POP0017,POP1864,POP65P
0,1,2025,BEAVER,7533,369,7163,2472,2071,4310,1152
1,1,2026,BEAVER,7577,351,7226,2516,2030,4353,1194
2,1,2027,BEAVER,7582,295,7287,2561,1996,4363,1223
3,1,2028,BEAVER,7634,303,7331,2600,1971,4386,1277
4,1,2029,BEAVER,7674,311,7363,2636,1946,4399,1329
...,...,...,...,...,...,...,...,...,...,...
1184,57,2061,WEBER,373889,5591,368298,163670,61768,223802,88319
1185,57,2062,WEBER,376857,5646,371211,165585,61525,225234,90098
1186,57,2063,WEBER,379822,5701,374122,167492,61327,226730,91765
1187,57,2064,WEBER,382790,5757,377033,169387,61184,228389,93217


In [6]:
# make df long with index CO_FIPS, YEAR
pop_hh_long_df = pop_hh_df.melt(
    id_vars=["CO_FIPS", "YEAR", "CO_NAME"],
    var_name="TDM_VARIABLE",
    value_name="VALUE"
)

# Reorder so CO_NAME is last
cols = ["CO_FIPS", "YEAR", "TDM_VARIABLE", "VALUE", "CO_NAME"]
pop_hh_long_df = pop_hh_long_df[cols]

pop_hh_long_df

,CO_FIPS,YEAR,TDM_VARIABLE,VALUE,CO_NAME
0,1,2025,TOTPOP,7533,BEAVER
1,1,2026,TOTPOP,7577,BEAVER
2,1,2027,TOTPOP,7582,BEAVER
3,1,2028,TOTPOP,7634,BEAVER
4,1,2029,TOTPOP,7674,BEAVER
...,...,...,...,...,...
8318,57,2061,POP65P,88319,WEBER
8319,57,2062,POP65P,90098,WEBER
8320,57,2063,POP65P,91765,WEBER
8321,57,2064,POP65P,93217,WEBER


In [7]:
pop_hh_long_df.to_csv('intermediate/tdm_population_by_county_2025+.csv', index=False)

In [8]:
# get Q
pop_hh_long_df

,CO_FIPS,YEAR,TDM_VARIABLE,VALUE,CO_NAME
0,1,2025,TOTPOP,7533,BEAVER
1,1,2026,TOTPOP,7577,BEAVER
2,1,2027,TOTPOP,7582,BEAVER
3,1,2028,TOTPOP,7634,BEAVER
4,1,2029,TOTPOP,7674,BEAVER
...,...,...,...,...,...
8318,57,2061,POP65P,88319,WEBER
8319,57,2062,POP65P,90098,WEBER
8320,57,2063,POP65P,91765,WEBER
8321,57,2064,POP65P,93217,WEBER
